# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
# Load the libraries as required.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import shap
import pickle

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
fires_dt['area_log'] = np.log1p(fires_dt['area'])

# Features and target
X = fires_dt.drop(columns=['area', 'area_log'])
y = fires_dt['area_log']

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
X.head()

Feature matrix shape: (517, 12)
Target shape: (517,)


,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (413, 12), Test: (104, 12)


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [5]:
# Identify numeric and categorical columns
num_cols = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
cat_cols = ['month', 'day']

# Preproc 1: StandardScaler for numerics + OneHotEncoder for categoricals
preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)

print("preproc1 defined: StandardScaler + OneHotEncoder")

preproc1 defined: StandardScaler + OneHotEncoder


### Preproc 2

Create preproc2 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [6]:
# Preproc 2: log1p transform on right-skewed FWI variables (dmc, dc, rain) before scaling.
# These variables are non-negative and heavily right-skewed, making log1p appropriate.
# The remaining numeric variables are only scaled.

skewed_cols  = ['dmc', 'dc', 'rain']          # highly skewed — apply log1p then scale
regular_cols = ['coord_x', 'coord_y', 'ffmc', 'isi', 'temp', 'rh', 'wind']  # scale only

log_then_scale = Pipeline([
    ('log1p', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

preproc2 = ColumnTransformer(
    transformers=[
        ('num_skewed',  log_then_scale,                                              skewed_cols),
        ('num_regular', StandardScaler(),                                            regular_cols),
        ('cat',         OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)

print("preproc2 defined: log1p + StandardScaler for skewed cols, StandardScaler for rest, OneHotEncoder for cats")

preproc2 defined: log1p + StandardScaler for skewed cols, StandardScaler for rest, OneHotEncoder for cats


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [7]:
# Pipeline A = preproc1 + baseline
pipeline_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])
print("Pipeline A: preproc1 + Ridge")

Pipeline A: preproc1 + Ridge


In [8]:
# Pipeline B = preproc2 + baseline
pipeline_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', Ridge())
])
print("Pipeline B: preproc2 + Ridge")

Pipeline B: preproc2 + Ridge


In [9]:
# Pipeline C = preproc1 + advanced model
pipeline_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', GradientBoostingRegressor(random_state=42))
])
print("Pipeline C: preproc1 + GradientBoostingRegressor")

Pipeline C: preproc1 + GradientBoostingRegressor


In [10]:
# Pipeline D = preproc2 + advanced model
pipeline_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', GradientBoostingRegressor(random_state=42))
])
print("Pipeline D: preproc2 + GradientBoostingRegressor")

Pipeline D: preproc2 + GradientBoostingRegressor


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [11]:
# Metric: MAE — a prediction-error metric (not a correlation metric like R²).
# neg_mean_absolute_error is used so GridSearchCV maximises (higher = better).
SCORING = 'neg_mean_absolute_error'
CV = 5  # 5-fold cross-validation

# --- Pipeline A ---
param_grid_A = {'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
gs_A = GridSearchCV(pipeline_A, param_grid_A, scoring=SCORING, cv=CV, n_jobs=-1)
gs_A.fit(X_train, y_train)
print(f"Pipeline A best params: {gs_A.best_params_}")
print(f"Pipeline A best CV MAE: {-gs_A.best_score_:.4f}")

Pipeline A best params: {'regressor__alpha': 100.0}
Pipeline A best CV MAE: 1.1674


In [12]:
# --- Pipeline B ---
param_grid_B = {'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
gs_B = GridSearchCV(pipeline_B, param_grid_B, scoring=SCORING, cv=CV, n_jobs=-1)
gs_B.fit(X_train, y_train)
print(f"Pipeline B best params: {gs_B.best_params_}")
print(f"Pipeline B best CV MAE: {-gs_B.best_score_:.4f}")

Pipeline B best params: {'regressor__alpha': 100.0}
Pipeline B best CV MAE: 1.1586


In [13]:
# --- Pipeline C ---
param_grid_C = {
    'regressor__n_estimators':  [100, 200],
    'regressor__max_depth':     [2, 3],
    'regressor__learning_rate': [0.05, 0.1]
}
gs_C = GridSearchCV(pipeline_C, param_grid_C, scoring=SCORING, cv=CV, n_jobs=-1)
gs_C.fit(X_train, y_train)
print(f"Pipeline C best params: {gs_C.best_params_}")
print(f"Pipeline C best CV MAE: {-gs_C.best_score_:.4f}")

Pipeline C best params: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 2, 'regressor__n_estimators': 100}
Pipeline C best CV MAE: 1.1680


In [14]:
# --- Pipeline D ---
param_grid_D = {
    'regressor__n_estimators':  [100, 200],
    'regressor__max_depth':     [2, 3],
    'regressor__learning_rate': [0.05, 0.1]
}
gs_D = GridSearchCV(pipeline_D, param_grid_D, scoring=SCORING, cv=CV, n_jobs=-1)
gs_D.fit(X_train, y_train)
print(f"Pipeline D best params: {gs_D.best_params_}")
print(f"Pipeline D best CV MAE: {-gs_D.best_score_:.4f}")

Pipeline D best params: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 2, 'regressor__n_estimators': 100}
Pipeline D best CV MAE: 1.1677


# Evaluate

+ Which model has the best performance?

In [15]:
# Evaluate all four best estimators on the held-out test set
results = {}
for name, gs in [('A (preproc1 + Ridge)', gs_A),
                 ('B (preproc2 + Ridge)', gs_B),
                 ('C (preproc1 + GBR)',   gs_C),
                 ('D (preproc2 + GBR)',   gs_D)]:
    y_pred = gs.best_estimator_.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'CV_MAE': -gs.best_score_}

results_df = pd.DataFrame(results).T.round(4)
print(results_df.sort_values('MAE'))

                         MAE    RMSE  CV_MAE
B (preproc2 + Ridge)  1.1913  1.4694  1.1586
A (preproc1 + Ridge)  1.1915  1.4724  1.1674
D (preproc2 + GBR)    1.1929  1.4846  1.1677
C (preproc1 + GBR)    1.1933  1.4848  1.1680


# Export

+ Save the best performing model to a pickle file.

In [16]:
# Identify the best model based on test MAE
best_name = results_df['MAE'].astype(float).idxmin()
gs_map = {
    'A (preproc1 + Ridge)': gs_A,
    'B (preproc2 + Ridge)': gs_B,
    'C (preproc1 + GBR)':   gs_C,
    'D (preproc2 + GBR)':   gs_D
}
best_model = gs_map[best_name].best_estimator_
print(f"Best model: {best_name}")

Best model: B (preproc2 + Ridge)


In [17]:
# Save best model as a pickle file
pickle_path = '../../05_src/data/fires/best_model.pkl'
with open(pickle_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"Model saved to {pickle_path}")

# Verify by reloading
with open(pickle_path, 'rb') as f:
    loaded_model = pickle.load(f)
print("Reload successful. Test MAE (reloaded):",
      round(mean_absolute_error(y_test, loaded_model.predict(X_test)), 4))

Model saved to ../../05_src/data/fires/best_model.pkl
Reload successful. Test MAE (reloaded): 1.1913


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

### Feature removal strategy

**Which features to remove?**  
From the global SHAP summary plot, features with mean |SHAP| values close to zero contribute little to predictions. Candidates for removal are typically `rain` (near-zero variance — almost always 0 mm) and `coord_y` (spatial coordinate with low predictive signal compared to weather indices). The one-hot encoded day-of-week columns also tend to have low global importance.

**Why?**  
Low-importance features add noise, increase model complexity, and can slow down grid search. For linear models like Ridge, they can also introduce unnecessary collinearity via correlated dummies.

**How to test whether removal improves performance?**  
1. Refit the best pipeline with the low-importance features dropped.
2. Compare cross-validated MAE (5-fold) between the full and reduced models.
3. If the reduced model's CV MAE is equal or lower, removal is justified.
4. Optionally, use `sklearn`'s `RFECV` (Recursive Feature Elimination with CV) to automate this process systematically.

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.